✅ 4. ParentDocumentRetriever

Use Case: Retrieve parent chunks based on a retrieval of smaller child chunks, great for keeping context intact in long documents.

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
#from langchain.schema.document import Document             no longer available
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore


import os
from dotenv import load_dotenv
# 2️⃣ Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")


# 1. Prepare documents
docs = [Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
        Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})]

# 2. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 3. Setup vectorstore for children
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=google_api_key
)

#Initialize empty FAISS vectorstore
vectorstore = FAISS.from_texts(
    ["temporary"],
    embedding=embedding
)

# 4. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),  # Stores parent docs
    child_splitter=child_splitter
)

# 5. Add documents
retriever.add_documents(docs)

# 6. Retrieve
results = retriever.invoke("What are agents?")

for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)


C:\Users\user\AppData\Local\Temp\ipykernel_11796\3831949372.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


📄 Retrieved Doc: Agents in LangChain use tools to answer questions.
📄 Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.


✅ 5. BM25Retriever

Use Case: Pure keyword-based search (like traditional search engines). No embeddings needed.

In [2]:
#!pip install rank_bm25

In [3]:
#from langchain.retrievers import BM25Retriever
#from langchain.schema.document import Document    both are no longer

from langchain_classic.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create simple text docs
docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(docs)

# Retrieve

results = bm25_retriever.invoke("How does BM25 work?")

for doc in results:
    print("📝 BM25 Result:", doc.page_content)


📝 BM25 Result: BM25 is a classical retrieval method.
📝 BM25 Result: Vector search is powerful.
📝 BM25 Result: LangChain enables LLM applications.


✅ 6. EnsembleRetriever

Use Case: Combine multiple retrievers (e.g., keyword + vector-based) with weighted scores.

In [4]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Sample docs
docs = [Document(page_content="LangChain supports LLMs."), Document(page_content="You can build AI apps using LangChain.")]

# BM25 Retriever
bm25 = BM25Retriever.from_documents(docs)

# Vector Retriever
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")
vectorstore = FAISS.from_documents(docs, embedding=embedding)

vector_retriever = vectorstore.as_retriever()

# Ensemble Retriever (equal weight)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

# Query
results = ensemble_retriever.invoke("AI apps using LangChain")
for doc in results:
    print("🔍 Ensemble Doc:", doc.page_content)


🔍 Ensemble Doc: You can build AI apps using LangChain.
🔍 Ensemble Doc: LangChain supports LLMs.


7.TimeWeightedVectorStoreRetriever

#This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).

In [5]:
#7.TimeWeightedVectorStoreRetriever
#This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
#from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime
import os

# Initialize embedding & vectorstore
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=google_api_key
)

#Initialize empty FAISS vectorstore
vectorstore = FAISS.from_texts(
    ["temporary"],
    embedding=embedding
)

docs = [
    Document(page_content="LangChain is for LLM-based apps", metadata={"last_accessed_at": datetime.now()}),
    Document(page_content="Vector search improves relevance", metadata={"last_accessed_at": datetime.now()})
]
vectorstore = FAISS.from_documents(docs, embedding)

# TimeWeighted Retriever
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

# Retrieve
results = retriever.invoke("What is LangChain?")
for r in results:
    print(r)
    print(r.page_content)


In [6]:
#!pip install tavily-python

8.TavilySearchAPIRetriever

In [7]:
from langchain_community.retrievers import TavilySearchAPIRetriever
import os

# Set your Tavily API key
#os.environ["TAVILY_API_KEY"] = "your_tavily_api_key"

from dotenv import load_dotenv
import os
# 🔐 Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Initialize Tavily Retriever
retriever = TavilySearchAPIRetriever(k=3)

# Perform search
docs = retriever.invoke("Latest updates about LangChain")
for doc in docs:
    print(doc.page_content)


More recently, in the latest v1.1 minor release, LangChain also added a model retry middleware with configurable exponential backoff, allowing graceful recovery for transient endpoint errors.

I personally think middleware is a game changer as agentic workflows get more complex, long-running, and stateful, especially when you need fine-grained control or robust error handling.
New integration packages for pluggable sandboxes: `langchain-modal`, `langchain-daytona`, and `langchain-runloop`. See sandboxes guide and example data analysis tutorial.
 Changes to conversation history summarization:
  + Summarization now happens in the model node via `wrap_model_call` events. Due to this we retain the full message history in the graph state.
  + More accurate token counting. [...] + Summarization will now automatically trigger if a chat model raises a `ContextOverflowError` (defined in `langchain-core`). Currently `langchain-anthropic` and `langchain-openai` support this.
 We now default to th